# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [129]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [165]:
from pathlib import Path

import numpy as np
import pandas as pd

FILE_NAME = "loo_summary_merfish"
CSV_PATH = f"../results/{FILE_NAME}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,200,0.068864,0.230110,0.115,0.217391,0.025,0.019022,14.429766,14.971449,521.428223
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,200,0.460826,0.497896,0.165,0.696970,0.115,0.293658,23.868064,27.635377,1234.587036
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,200,0.399375,0.309090,0.210,0.857143,0.180,0.069700,17.152134,18.944778,1589.633545
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,200,0.701164,0.642562,0.280,0.928571,0.260,0.000000,36.351183,38.485640,2399.222656
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,200,0.265527,0.311898,0.185,0.945946,0.175,0.003236,17.114140,17.957415,1034.296997


In [167]:
# Rename holdout_celltype by replacing the last '_' in with '-'
df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)

In [168]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,200,0.068864,0.230110,0.115,0.217391,0.025,0.019022,14.429766,14.971449,521.428223,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,200,0.460826,0.497896,0.165,0.696970,0.115,0.293658,23.868064,27.635377,1234.587036,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,200,0.399375,0.309090,0.210,0.857143,0.180,0.069700,17.152134,18.944778,1589.633545,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,200,0.701164,0.642562,0.280,0.928571,0.260,0.000000,36.351183,38.485640,2399.222656,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,200,0.265527,0.311898,0.185,0.945946,0.175,0.003236,17.114140,17.957415,1034.296997,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,C57BL6J-2.041,scgen,endothelial cell,200,0.890137,0.905810,0.505,1.000000,0.505,0.905823,2.770841,1.605527,1262.424805,Isocortex
146,C57BL6J-2.041,scgen,glutamatergic neuron,200,0.563459,0.582064,0.230,0.891304,0.205,0.742252,8.986255,10.216734,6664.635742,Fiber-tracts
147,C57BL6J-2.041,scgen,glutamatergic neuron,200,0.730424,0.820496,0.310,1.000000,0.310,0.448907,11.368176,14.479038,8145.387695,Isocortex
148,C57BL6J-2.041,scgen,oligodendrocyte,200,0.417180,0.721177,0.495,0.979798,0.485,0.668434,1.197481,2.115432,2080.420166,Fiber-tracts


In [169]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "direction_match_k": +1,
    "edistance_local": -1,
    #"rmse": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    #"mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    #"cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    #"MintFlow",
    "cellina (ablated)",
    #"cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'CPA', 'scGen'],
      dtype=object)

In [170]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [171]:
agg

spearman             pearson            \
                                    mean       std      mean       std   
perturbation model_name                                                  
Fiber-tracts mean shift         0.216837  0.202377  0.217113  0.177359   
             CPA                0.683510  0.093951  0.700696  0.139890   
             scGen              0.609789  0.097364  0.650931  0.121530   
             cellina (ablated)  0.585799  0.098770  0.621369  0.150721   
             cellina            0.738202  0.121254  0.740552  0.155519   
Isocortex    mean shift         0.578568  0.164632  0.506310  0.156934   
             CPA                0.836440  0.089843  0.815121  0.183016   
             scGen              0.799701  0.079724  0.791956  0.143871   
             cellina (ablated)  0.755767  0.112528  0.724356  0.172904   
             cellina            0.864166  0.074786  0.842079  0.175579   

                               precision           direction_match_k  \
                                    mean       std              mean   
perturbation model_name                                                
Fiber-tracts mean shift         0.214000  0.043924          0.145000   
             CPA                0.364667  0.107844          0.331333   
             scGen              0.360667  0.081215          0.343333   
             cellina (ablated)  0.285667  0.108691          0.247667   
             cellina            0.442667  0.114811          0.414000   
Isocortex    mean shift         0.284333  0.056942          0.243333   
             CPA                0.473000  0.078622          0.467333   
             scGen              0.407667  0.066919          0.404000   
             cellina (ablated)  0.346333  0.111058          0.333000   
             cellina            0.532000  0.061377          0.528333   

                                         edistance_local            
                                     std            mean       std  
perturbation model_name                                             
Fiber-tracts mean shift         0.060651       20.540958  3.055594  
             CPA                0.113002        7.668568  5.392994  
             scGen              0.086016        5.670908  4.245609  
             cellina (ablated)  0.092772        8.207118  5.756282  
             cellina            0.125871        7.744071  5.205701  
Isocortex    mean shift         0.077912       34.584805  6.660973  
             CPA                0.082003        8.721271  6.553784  
             scGen              0.067829        7.702896  5.985113  
             cellina (ablated)  0.111079        9.895056  7.512041  
             cellina            0.066377        8.948595  6.450584

In [172]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s

In [173]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.217 $\pm$ 0.202   
             CPA                         0.684 $\pm$ 0.094   
             scGen                       0.610 $\pm$ 0.097   
             cellina (ablated)           0.586 $\pm$ 0.099   
             cellina            \textbf{0.738 $\pm$ 0.121}   
Isocortex    mean shift                  0.579 $\pm$ 0.165   
             CPA                         0.836 $\pm$ 0.090   
             scGen                       0.800 $\pm$ 0.080   
             cellina (ablated)           0.756 $\pm$ 0.113   
             cellina            \textbf{0.864 $\pm$ 0.075}   

                                        Pearson $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.217 $\pm$ 0.177   
             CPA                         0.701 $\pm$ 0.140   
             scGen                       0.651 $\pm$ 0.122   
             cellina (ablated)           0.621 $\pm$ 0.151   
             cellina            \textbf{0.741 $\pm$ 0.156}   
Isocortex    mean shift                  0.506 $\pm$ 0.157   
             CPA                         0.815 $\pm$ 0.183   
             scGen                       0.792 $\pm$ 0.144   
             cellina (ablated)           0.724 $\pm$ 0.173   
             cellina            \textbf{0.842 $\pm$ 0.176}   

                                      Precision $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.214 $\pm$ 0.044   
             CPA                         0.365 $\pm$ 0.108   
             scGen                       0.361 $\pm$ 0.081   
             cellina (ablated)           0.286 $\pm$ 0.109   
             cellina            \textbf{0.443 $\pm$ 0.115}   
Isocortex    mean shift                  0.284 $\pm$ 0.057   
             CPA                         0.473 $\pm$ 0.079   
             scGen                       0.408 $\pm$ 0.067   
             cellina (ablated)           0.346 $\pm$ 0.111   
             cellina            \textbf{0.532 $\pm$ 0.061}   

                               Direction Match (K) $\uparrow$  \
perturbation model_name                                         
Fiber-tracts mean shift                     0.145 $\pm$ 0.061   
             CPA                            0.331 $\pm$ 0.113   
             scGen                          0.343 $\pm$ 0.086   
             cellina (ablated)              0.248 $\pm$ 0.093   
             cellina               \textbf{0.414 $\pm$ 0.126}   
Isocortex    mean shift                     0.243 $\pm$ 0.078   
             CPA                            0.467 $\pm$ 0.082   
             scGen                          0.404 $\pm$ 0.068   
             cellina (ablated)              0.333 $\pm$ 0.111   
             cellina               \textbf{0.528 $\pm$ 0.066}   

                               E-dist (local) $\downarrow$  
perturbation model_name                                     
Fiber-tracts mean shift                 20.541 $\pm$ 3.056  
             CPA                         7.669 $\pm$ 5.393  
             scGen              \textbf{5.671 $\pm$ 4.246}  
             cellina (ablated)           8.207 $\pm$ 5.756  
             cellina                     7.744 $\pm$ 5.206  
Isocortex    mean shift                 34.585 $\pm$ 6.661  
             CPA                         8.721 $\pm$ 6.554  
             scGen              \textbf{7.703 $\pm$ 5.985}  
             cellina (ablated)           9.895 $\pm$ 7.512  
             cellina                     8.949 $\pm$ 6.451

In [174]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:{FILE_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:{FILE_NAME}",
    )

print(latex)

(OUT_DIR / f"{FILE_NAME}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 200 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{llccccc}
\toprule
Holdout \\ perturbation & Method & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & Direction Match (K) $\uparrow$ & E-dist (local) $\downarrow$ \\
\toprule

\multirow[c]{5}{*}{Fiber-tracts} & mean shift & 0.217 $\pm$ 0.202 & 0.217 $\pm$ 0.177 & 0.214 $\pm$ 0.044 & 0.145 $\pm$ 0.061 & 20.541 $\pm$ 3.056 \\
 & CPA & 0.684 $\pm$ 0.094 & 0.701 $\pm$ 0.140 & 0.365 $\pm$ 0.108 & 0.331 $\pm$ 0.113 & 7.669 $\pm$ 5.393 \\
 & scGen & 0.610 $\pm$ 0.097 & 0.651 $\pm$ 0.122 & 0.361 $\pm$ 0.081 & 0.343 $\pm$ 0.086 & \textbf{5.671 $\pm$ 4.246} \\
 & cellina (ablated) & 0.586 $\pm$ 0.099 & 0.621 $\pm$ 0.151 & 0.286 $\pm$ 0.109 & 0.248 $\pm$ 0.093 & 8.207 $\pm$ 5.756 \\
 & cellina & \textbf{0.73

1817

In [175]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                                | Spearman ↑    | Pearson ↑     | Precision ↑   | Direction Match (K) ↑   | E-dist (local) ↓   |
|:--------------------------------------|:--------------|:--------------|:--------------|:------------------------|:-------------------|
| ('Fiber-tracts', 'mean shift')        | 0.217 ± 0.202 | 0.217 ± 0.177 | 0.214 ± 0.044 | 0.145 ± 0.061           | 20.541 ± 3.056     |
| ('Fiber-tracts', 'CPA')               | 0.684 ± 0.094 | 0.701 ± 0.140 | 0.365 ± 0.108 | 0.331 ± 0.113           | 7.669 ± 5.393      |
| ('Fiber-tracts', 'scGen')             | 0.610 ± 0.097 | 0.651 ± 0.122 | 0.361 ± 0.081 | 0.343 ± 0.086           | 5.671 ± 4.246      |
| ('Fiber-tracts', 'cellina (ablated)') | 0.586 ± 0.099 | 0.621 ± 0.151 | 0.286 ± 0.109 | 0.248 ± 0.093           | 8.207 ± 5.756      |
| ('Fiber-tracts', 'cellina')           | 0.738 ± 0.121 | 0.741 ± 0.156 | 0.443 ± 0.115 | 0.414 ± 0.126           | 7.744 ± 5.206      |
| ('Isocortex', 'mean shift')           |